# 05 · Symptom prediction (leak-free)

Visualises `src/symptom_prediction` output (run `python -m src.symptom_prediction.leakage_audit` and
`python -m src.symptom_prediction.run`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.core import paths
from src.core.data import GROUPS, load_data, load_model_results, load_features

sns.set_theme(style="whitegrid", context="notebook")
PALETTE = {"HC": "#2a9d8f", "AP": "#e76f51", "NAP": "#e9c46a"}

## Leakage audit of the v1 PANSS model

In [ ]:
D = paths.DERIVED / "symptom_prediction"
L = pd.read_csv(D / "leakage_audit.csv"); null = pd.read_csv(D / "leakage_null.csv")
display(L.round(4))
leaky = L[(L.threshold == 0.2) & L.selection.str.startswith("full")].R2.item()
nested = L[(L.threshold == 0.2) & (L.selection == "inside folds")].R2.item()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null.R2, bins=30, color="lightgray", label="leaky pipeline, shuffled PANSS")
ax.axvline(leaky, color="crimson", lw=2, label=f"v1 leaky R² = {leaky:.2f}")
ax.axvline(nested, color="navy", lw=2, label=f"selection inside folds R² = {nested:.2f}")
ax.set_xlabel("LOOCV R²"); ax.legend()
plt.tight_layout(); plt.show()

## Model comparison

`r_within`: CV prediction vs truth after removing AP/NAP means (averaged over 5 repeats). Grey bar = 95th percentile
of the within-group permutation null; negative values are a CV shrinkage artefact shared by the null.

In [ ]:
R = pd.read_csv(D / "model_comparison.csv")
display(R.round(3))
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, t in zip(axes, ("PANSS", "YMRS")):
    d = R[R.target == t].reset_index(drop=True)
    y = np.arange(len(d))
    ax.barh(y, d.null_95 - d.null_mean, left=d.null_mean, color="lightgray", label="null mean → 95th pct")
    ax.scatter(d.r_within, y, color="crimson", zorder=3, label="observed")
    for i, p in enumerate(d.p_perm): ax.text(max(d.r_within.max(), d.null_95.max()) + 0.02, i, f"p={p:.2f}", va="center", fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels(d.features); ax.set_title(t); ax.set_xlabel("within-group r")
axes[0].legend(loc="lower left")
plt.tight_layout(); plt.show()